In [1]:
import os
import torch
import numpy as np
from PIL import Image
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace Hub")
else:
    print("WARNING: HF_TOKEN not found in .env — some gated models may fail to load")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HuggingFace Hub
Using device: cpu


In [2]:
# Create a dummy 224x224 RGB image as a PIL Image (required by all HF processors)
dummy_pil = Image.fromarray(
    (np.random.rand(224, 224, 3) * 255).astype(np.uint8)
)
print(f"Dummy image size: {dummy_pil.size}, mode: {dummy_pil.mode}")

# Helper to summarize hidden states from a forward pass
def inspect_hidden_states(outputs, backbone_name, expected_seq_len, expected_patch_count, layer=11):
    hs = outputs.hidden_states
    print(f"\n{'='*60}")
    print(f"  {backbone_name}")
    print(f"{'='*60}")
    print(f"  Total hidden state tensors returned : {len(hs)}")
    print(f"  (index 0 = embedding, indices 1..N = transformer blocks)")
    print()
    for i, h in enumerate(hs):
        label = "embedding" if i == 0 else f"block {i-1:>2}"
        print(f"  hidden_states[{i:>2}]  ({label})  shape: {tuple(h.shape)}")
    
    print()
    target_idx = layer + 1
    h_layer = hs[target_idx]
    seq_len = h_layer.shape[1]
    d = h_layer.shape[2]
    print(f"  >> Layer {layer} hidden state: hidden_states[{target_idx}]  shape={tuple(h_layer.shape)}")
    print(f"  >> Sequence length        : {seq_len}  (expected: {expected_seq_len})")
    print(f"  >> Hidden dim             : {d}  (expected: 768)")
    
    if seq_len != expected_seq_len:
        print(f"  ⚠️  MISMATCH: seq_len {seq_len} != expected {expected_seq_len}")
    if d != 768:
        print(f"  ⚠️  MISMATCH: d_model {d} != expected 768")
    
    num_special = seq_len - expected_patch_count
    print(f"  >> Special tokens inferred: {num_special}  (seq_len - expected_patches = {seq_len} - {expected_patch_count})")
    return h_layer

Dummy image size: (224, 224), mode: RGB


In [3]:
# ── CLIP ViT-B/16 ──────────────────────────────────────────────────────────────
from transformers import CLIPVisionModel, CLIPImageProcessor

clip_processor = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch16")
clip_model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch16").eval().to(device)

clip_inputs = clip_processor(images=dummy_pil, return_tensors="pt")
clip_inputs = {k: v.to(device) for k, v in clip_inputs.items()}

with torch.no_grad():
    clip_outputs = clip_model(**clip_inputs, output_hidden_states=True)

# CLIP: seq_len = 1 (CLS) + 196 (patches) = 197
clip_layer11 = inspect_hidden_states(clip_outputs, "CLIP ViT-B/16", expected_seq_len=197, expected_patch_count=196)

# Drop CLS (position 0) — patch tokens should be [1, 196, 768]
clip_patches = clip_layer11[:, 1:, :]
print(f"\n  After dropping CLS (pos 0): {tuple(clip_patches.shape)}")
if clip_patches.shape == (1, 196, 768):
    print("  ✅ Shape correct: [1, 196, 768]")
else:
    print(f"  ⚠️  Unexpected shape: {tuple(clip_patches.shape)}")


  CLIP ViT-B/16
  Total hidden state tensors returned : 13
  (index 0 = embedding, indices 1..N = transformer blocks)

  hidden_states[ 0]  (embedding)  shape: (1, 197, 768)
  hidden_states[ 1]  (block  0)  shape: (1, 197, 768)
  hidden_states[ 2]  (block  1)  shape: (1, 197, 768)
  hidden_states[ 3]  (block  2)  shape: (1, 197, 768)
  hidden_states[ 4]  (block  3)  shape: (1, 197, 768)
  hidden_states[ 5]  (block  4)  shape: (1, 197, 768)
  hidden_states[ 6]  (block  5)  shape: (1, 197, 768)
  hidden_states[ 7]  (block  6)  shape: (1, 197, 768)
  hidden_states[ 8]  (block  7)  shape: (1, 197, 768)
  hidden_states[ 9]  (block  8)  shape: (1, 197, 768)
  hidden_states[10]  (block  9)  shape: (1, 197, 768)
  hidden_states[11]  (block 10)  shape: (1, 197, 768)
  hidden_states[12]  (block 11)  shape: (1, 197, 768)

  >> Layer 11 hidden state: hidden_states[12]  shape=(1, 197, 768)
  >> Sequence length        : 197  (expected: 197)
  >> Hidden dim             : 768  (expected: 768)
  >> Sp

In [4]:
# ── DINOv2 ViT-B/14 ────────────────────────────────────────────────────────────
from transformers import Dinov2Model, AutoImageProcessor

dino_processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
dino_model = Dinov2Model.from_pretrained("facebook/dinov2-base").eval().to(device)

dino_inputs = dino_processor(images=dummy_pil, return_tensors="pt")
dino_inputs = {k: v.to(device) for k, v in dino_inputs.items()}

with torch.no_grad():
    dino_outputs = dino_model(**dino_inputs, output_hidden_states=True)

# DINOv2-base: seq_len = 1 (CLS) + 256 (patches) = 257 (no registers)
dino_layer11 = inspect_hidden_states(dino_outputs, "DINOv2 ViT-B/14 (facebook/dinov2-base)", expected_seq_len=257, expected_patch_count=256)

# Drop CLS (position 0) — patch tokens should be [1, 256, 768]
dino_patches = dino_layer11[:, 1:, :]
print(f"\n  After dropping CLS (pos 0): {tuple(dino_patches.shape)}")
if dino_patches.shape == (1, 256, 768):
    print("  ✅ Shape correct: [1, 256, 768]")
else:
    print(f"  ⚠️  Unexpected shape: {tuple(dino_patches.shape)}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



  DINOv2 ViT-B/14 (facebook/dinov2-base)
  Total hidden state tensors returned : 13
  (index 0 = embedding, indices 1..N = transformer blocks)

  hidden_states[ 0]  (embedding)  shape: (1, 257, 768)
  hidden_states[ 1]  (block  0)  shape: (1, 257, 768)
  hidden_states[ 2]  (block  1)  shape: (1, 257, 768)
  hidden_states[ 3]  (block  2)  shape: (1, 257, 768)
  hidden_states[ 4]  (block  3)  shape: (1, 257, 768)
  hidden_states[ 5]  (block  4)  shape: (1, 257, 768)
  hidden_states[ 6]  (block  5)  shape: (1, 257, 768)
  hidden_states[ 7]  (block  6)  shape: (1, 257, 768)
  hidden_states[ 8]  (block  7)  shape: (1, 257, 768)
  hidden_states[ 9]  (block  8)  shape: (1, 257, 768)
  hidden_states[10]  (block  9)  shape: (1, 257, 768)
  hidden_states[11]  (block 10)  shape: (1, 257, 768)
  hidden_states[12]  (block 11)  shape: (1, 257, 768)

  >> Layer 11 hidden state: hidden_states[12]  shape=(1, 257, 768)
  >> Sequence length        : 257  (expected: 257)
  >> Hidden dim             : 768

In [5]:
# ── SigLIP ViT-B/16 ────────────────────────────────────────────────────────────
from transformers import SiglipVisionModel, AutoProcessor

siglip_processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
siglip_model = SiglipVisionModel.from_pretrained("google/siglip-base-patch16-224").eval().to(device)

siglip_inputs = siglip_processor(images=dummy_pil, return_tensors="pt")
siglip_inputs = {k: v.to(device) for k, v in siglip_inputs.items()}

with torch.no_grad():
    siglip_outputs = siglip_model(**siglip_inputs, output_hidden_states=True)

# SigLIP: no CLS token — seq_len = 196 (all patches)
siglip_layer11 = inspect_hidden_states(siglip_outputs, "SigLIP ViT-B/16", expected_seq_len=196, expected_patch_count=196)

# No tokens to drop — all 196 are patch tokens
siglip_patches = siglip_layer11
print(f"\n  No special tokens to drop: {tuple(siglip_patches.shape)}")
if siglip_patches.shape == (1, 196, 768):
    print("  ✅ Shape correct: [1, 196, 768]")
else:
    print(f"  ⚠️  Unexpected shape: {tuple(siglip_patches.shape)}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



  SigLIP ViT-B/16
  Total hidden state tensors returned : 13
  (index 0 = embedding, indices 1..N = transformer blocks)

  hidden_states[ 0]  (embedding)  shape: (1, 196, 768)
  hidden_states[ 1]  (block  0)  shape: (1, 196, 768)
  hidden_states[ 2]  (block  1)  shape: (1, 196, 768)
  hidden_states[ 3]  (block  2)  shape: (1, 196, 768)
  hidden_states[ 4]  (block  3)  shape: (1, 196, 768)
  hidden_states[ 5]  (block  4)  shape: (1, 196, 768)
  hidden_states[ 6]  (block  5)  shape: (1, 196, 768)
  hidden_states[ 7]  (block  6)  shape: (1, 196, 768)
  hidden_states[ 8]  (block  7)  shape: (1, 196, 768)
  hidden_states[ 9]  (block  8)  shape: (1, 196, 768)
  hidden_states[10]  (block  9)  shape: (1, 196, 768)
  hidden_states[11]  (block 10)  shape: (1, 196, 768)
  hidden_states[12]  (block 11)  shape: (1, 196, 768)

  >> Layer 11 hidden state: hidden_states[12]  shape=(1, 196, 768)
  >> Sequence length        : 196  (expected: 196)
  >> Hidden dim             : 768  (expected: 768)
  >> 

In [6]:
# ── MAE ViT-B/16 ───────────────────────────────────────────────────────────────
from transformers import ViTMAEModel, AutoImageProcessor as MAEImageProcessor

mae_processor = MAEImageProcessor.from_pretrained("facebook/vit-mae-base")
mae_model = ViTMAEModel.from_pretrained("facebook/vit-mae-base")
# Disable masking so all 196 patches are visible (seq_len = 1 + 196 = 197)
mae_model.config.mask_ratio = 0.0
mae_model = mae_model.eval().to(device)

mae_inputs = mae_processor(images=dummy_pil, return_tensors="pt")
mae_inputs = {k: v.to(device) for k, v in mae_inputs.items()}

with torch.no_grad():
    mae_outputs = mae_model(**mae_inputs, output_hidden_states=True)

# MAE with mask_ratio=0.0: seq_len should be 1 (CLS) + 196 (patches) = 197
mae_layer11 = inspect_hidden_states(mae_outputs, "MAE ViT-B/16 (facebook/vit-mae-base)", expected_seq_len=197, expected_patch_count=196)

# Drop CLS (position 0) — patch tokens should be [1, 196, 768]
mae_patches = mae_layer11[:, 1:, :]
print(f"\n  After dropping CLS (pos 0): {tuple(mae_patches.shape)}")
if mae_patches.shape == (1, 196, 768):
    print("  ✅ Shape correct: [1, 196, 768]")
else:
    print(f"  ⚠️  Unexpected shape: {tuple(mae_patches.shape)}")

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.



  MAE ViT-B/16 (facebook/vit-mae-base)
  Total hidden state tensors returned : 13
  (index 0 = embedding, indices 1..N = transformer blocks)

  hidden_states[ 0]  (embedding)  shape: (1, 197, 768)
  hidden_states[ 1]  (block  0)  shape: (1, 197, 768)
  hidden_states[ 2]  (block  1)  shape: (1, 197, 768)
  hidden_states[ 3]  (block  2)  shape: (1, 197, 768)
  hidden_states[ 4]  (block  3)  shape: (1, 197, 768)
  hidden_states[ 5]  (block  4)  shape: (1, 197, 768)
  hidden_states[ 6]  (block  5)  shape: (1, 197, 768)
  hidden_states[ 7]  (block  6)  shape: (1, 197, 768)
  hidden_states[ 8]  (block  7)  shape: (1, 197, 768)
  hidden_states[ 9]  (block  8)  shape: (1, 197, 768)
  hidden_states[10]  (block  9)  shape: (1, 197, 768)
  hidden_states[11]  (block 10)  shape: (1, 197, 768)
  hidden_states[12]  (block 11)  shape: (1, 197, 768)

  >> Layer 11 hidden state: hidden_states[12]  shape=(1, 197, 768)
  >> Sequence length        : 197  (expected: 197)
  >> Hidden dim             : 768  

In [7]:
# ── DeiT ViT-B/16 ──────────────────────────────────────────────────────────────
from transformers import DeiTModel, DeiTImageProcessor

deit_processor = DeiTImageProcessor.from_pretrained("facebook/deit-base-patch16-224")
deit_model = DeiTModel.from_pretrained("facebook/deit-base-patch16-224").eval().to(device)

deit_inputs = deit_processor(images=dummy_pil, return_tensors="pt")
deit_inputs = {k: v.to(device) for k, v in deit_inputs.items()}

with torch.no_grad():
    deit_outputs = deit_model(**deit_inputs, output_hidden_states=True)

# DeiT: seq_len = 1 (CLS) + 1 (dist) + 196 (patches) = 198
deit_layer11 = inspect_hidden_states(deit_outputs, "DeiT ViT-B/16", expected_seq_len=198, expected_patch_count=196)

# Drop CLS (pos 0) AND distillation token (pos 1) — patch tokens should be [1, 196, 768]
deit_patches = deit_layer11[:, 2:, :]
print(f"\n  After dropping CLS (pos 0) + dist (pos 1): {tuple(deit_patches.shape)}")
if deit_patches.shape == (1, 196, 768):
    print("  ✅ Shape correct: [1, 196, 768]")
else:
    print(f"  ⚠️  Unexpected shape: {tuple(deit_patches.shape)}")

You are using a model of type vit to instantiate a model of type deit. This is not supported for all configurations of models and can yield errors.
Some weights of DeiTModel were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized: ['embeddings.cls_token', 'embeddings.distillation_token', 'embeddings.patch_embeddings.projection.bias', 'embeddings.patch_embeddings.projection.weight', 'embeddings.position_embeddings', 'encoder.layer.0.attention.attention.key.bias', 'encoder.layer.0.attention.attention.key.weight', 'encoder.layer.0.attention.attention.query.bias', 'encoder.layer.0.attention.attention.query.weight', 'encoder.layer.0.attention.attention.value.bias', 'encoder.layer.0.attention.attention.value.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.layernorm_after.bias', 'enc


  DeiT ViT-B/16
  Total hidden state tensors returned : 13
  (index 0 = embedding, indices 1..N = transformer blocks)

  hidden_states[ 0]  (embedding)  shape: (1, 198, 768)
  hidden_states[ 1]  (block  0)  shape: (1, 198, 768)
  hidden_states[ 2]  (block  1)  shape: (1, 198, 768)
  hidden_states[ 3]  (block  2)  shape: (1, 198, 768)
  hidden_states[ 4]  (block  3)  shape: (1, 198, 768)
  hidden_states[ 5]  (block  4)  shape: (1, 198, 768)
  hidden_states[ 6]  (block  5)  shape: (1, 198, 768)
  hidden_states[ 7]  (block  6)  shape: (1, 198, 768)
  hidden_states[ 8]  (block  7)  shape: (1, 198, 768)
  hidden_states[ 9]  (block  8)  shape: (1, 198, 768)
  hidden_states[10]  (block  9)  shape: (1, 198, 768)
  hidden_states[11]  (block 10)  shape: (1, 198, 768)
  hidden_states[12]  (block 11)  shape: (1, 198, 768)

  >> Layer 11 hidden state: hidden_states[12]  shape=(1, 198, 768)
  >> Sequence length        : 198  (expected: 198)
  >> Hidden dim             : 768  (expected: 768)
  >> Sp

In [8]:
# ── Summary Table ──────────────────────────────────────────────────────────────
results = [
    {
        "backbone":           "CLIP ViT-B/16",
        "model_id":           "openai/clip-vit-base-patch16",
        "total_hs":           len(clip_outputs.hidden_states),
        "seq_len_layer11":    clip_layer11.shape[1],
        "special_tokens":     "CLS (pos 0)",
        "patch_shape":        str(tuple(clip_patches.shape)),
        "expected_shape":     "(1, 196, 768)",
        "pass":               clip_patches.shape == (1, 196, 768),
    },
    {
        "backbone":           "DINOv2 ViT-B/14",
        "model_id":           "facebook/dinov2-base",
        "total_hs":           len(dino_outputs.hidden_states),
        "seq_len_layer11":    dino_layer11.shape[1],
        "special_tokens":     "CLS (pos 0)",
        "patch_shape":        str(tuple(dino_patches.shape)),
        "expected_shape":     "(1, 256, 768)",
        "pass":               dino_patches.shape == (1, 256, 768),
    },
    {
        "backbone":           "SigLIP ViT-B/16",
        "model_id":           "google/siglip-base-patch16-224",
        "total_hs":           len(siglip_outputs.hidden_states),
        "seq_len_layer11":    siglip_layer11.shape[1],
        "special_tokens":     "None",
        "patch_shape":        str(tuple(siglip_patches.shape)),
        "expected_shape":     "(1, 196, 768)",
        "pass":               siglip_patches.shape == (1, 196, 768),
    },
    {
        "backbone":           "MAE ViT-B/16",
        "model_id":           "facebook/vit-mae-base",
        "total_hs":           len(mae_outputs.hidden_states),
        "seq_len_layer11":    mae_layer11.shape[1],
        "special_tokens":     "CLS (pos 0)",
        "patch_shape":        str(tuple(mae_patches.shape)),
        "expected_shape":     "(1, 196, 768)",
        "pass":               mae_patches.shape[0] == 1 and mae_patches.shape == (1, 196, 768)
    },
    {
        "backbone":           "DeiT ViT-B/16",
        "model_id":           "facebook/deit-base-patch16-224",
        "total_hs":           len(deit_outputs.hidden_states),
        "seq_len_layer11":    deit_layer11.shape[1],
        "special_tokens":     "CLS (pos 0), Dist (pos 1)",
        "patch_shape":        str(tuple(deit_patches.shape)),
        "expected_shape":     "(1, 196, 768)",
        "pass":               deit_patches.shape == (1, 196, 768),
    },
]

print(f"\n{'='*100}")
print(f"{'BACKBONE VERIFICATION SUMMARY':^100}")
print(f"{'='*100}")
hdr = f"{'Backbone':<20} {'total_hs':>8} {'seq_len':>8} {'Special Tokens':<26} {'Patch Shape':<18} {'Expected':<18} {'Status':>6}"
print(hdr)
print("-" * 100)
for r in results:
    status = "✅ PASS" if r["pass"] else "⚠️  FAIL"
    print(
        f"{r['backbone']:<20} {r['total_hs']:>8} {r['seq_len_layer11']:>8} "
        f"{r['special_tokens']:<26} {r['patch_shape']:<18} {r['expected_shape']:<18} {status:>6}"
    )
print("="*100)


                                   BACKBONE VERIFICATION SUMMARY                                    
Backbone             total_hs  seq_len Special Tokens             Patch Shape        Expected           Status
----------------------------------------------------------------------------------------------------
CLIP ViT-B/16              13      197 CLS (pos 0)                (1, 196, 768)      (1, 196, 768)      ✅ PASS
DINOv2 ViT-B/14            13      257 CLS (pos 0)                (1, 256, 768)      (1, 256, 768)      ✅ PASS
SigLIP ViT-B/16            13      196 None                       (1, 196, 768)      (1, 196, 768)      ✅ PASS
MAE ViT-B/16               13      197 CLS (pos 0)                (1, 196, 768)      (1, 196, 768)      ✅ PASS
DeiT ViT-B/16              13      198 CLS (pos 0), Dist (pos 1)  (1, 196, 768)      (1, 196, 768)      ✅ PASS
